In [1]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report,average_precision_score
import optuna
from optuna.trial import Trial
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt

/Users/prathikkundaragi/.local/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
data = pd.read_parquet("/Users/prathikkundaragi/MY PROJECTS/Fraud Dataset/model_data_full.parquet")
data.head(5)

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,type_CASH_OUT,type_TRANSFER
2,181.00,181.0,0.0,0.0,0.00,1,0,1
3,181.00,181.0,0.0,21182.0,0.00,1,1,0
15,229133.94,15325.0,0.0,5083.0,51513.44,0,1,0
19,215310.30,705.0,0.0,22425.0,0.00,0,0,1
24,311685.89,10835.0,0.0,6267.0,2719172.89,0,0,1


In [3]:
#creating train and holdout datasets
data_train, data_holdout = train_test_split(data, test_size=0.3, random_state=42)

In [4]:
#creating X and y exogenous and endogenous variables for training
X_train = data_train.drop('isFraud', axis=1)
y_train = data_train['isFraud']

In [5]:
#importing the XGBOOST module and running the tuning function for the full dataset
import XGBOOST_module

XGBOOST_full= XGBOOST_module.run_fraud_tuning(X_train, y_train, n_trials=4)



  0%|          | 0/4 [00:00<?, ?it/s]

  CV   AUC-PR  (best trial) : 0.9218
  CV   KS Stat (best trial) : 0.9808
  Internal Test AUC-PR      : 0.9248
  Internal Test KS Stat     : 0.9839
  Best Params               : {'n_estimators': 450, 'max_depth': 10, 'learning_rate': 0.06504856968981275, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.4936111842654619, 'min_child_weight': 2, 'gamma': 0.2904180608409973, 'reg_alpha': 2.142302175774105, 'reg_lambda': 0.10129197956845731, 'scale_pos_weight': 109.13016089144637}
  Final model retrained on full X_train.
  Apply to holdout externally in your prediction notebook.


In [6]:
#creating holdout X and y exogenous and endogenous variables for testing
X_holdout = data_holdout.drop('isFraud', axis=1)
y_holdout = data_holdout['isFraud']

In [7]:
fraud_score= pd.DataFrame()
fraud_score.index= X_holdout.index
fraud_score['fraud_score'] =  XGBOOST_full['model'].predict_proba(X_holdout)[:, 1]
fraud_score['isFraud'] = y_holdout
fraud_score['amount'] = X_holdout['amount']
fraud_score.head(5)

,fraud_score,isFraud,amount
1442460,0.008572,0,285073.76
5847267,0.000085,0,204657.38
2163940,0.000312,0,27554.10
2689116,0.000016,0,100950.61
1452693,0.000090,0,42737.06


In [10]:
fraud_score.sort_values(by='fraud_score', ascending=False, inplace=True)
fraud_score.head(10)

,fraud_score,isFraud,amount,fraud_amount,score_bucket
6168504,0.999998,1,9421856.87,9421856.87,0.9-1.0
10396,0.999998,1,5460002.91,5460002.91,0.9-1.0
586312,0.999997,1,10000000.00,10000000.00,0.9-1.0
5996406,0.999997,1,10000000.00,10000000.00,0.9-1.0
4202744,0.999997,1,10000000.00,10000000.00,0.9-1.0
5563724,0.999997,1,10000000.00,10000000.00,0.9-1.0
6168498,0.999997,1,10000000.00,10000000.00,0.9-1.0
4441,0.999997,1,10000000.00,10000000.00,0.9-1.0
6296769,0.999997,1,10000000.00,10000000.00,0.9-1.0
3542878,0.999997,1,10000000.00,10000000.00,0.9-1.0


In [11]:
# Create fraud_amount column — zero for non-fraud rows
fraud_score['fraud_amount'] = fraud_score['amount'] * fraud_score['isFraud']

# Create score buckets
fraud_score['score_bucket'] = pd.cut(fraud_score['fraud_score'], 
                                      bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
                                      labels=['0-0.1', '0.1-0.2', '0.2-0.3', '0.3-0.4', '0.4-0.5', 
                                             '0.5-0.6', '0.6-0.7', '0.7-0.8', '0.8-0.9', '0.9-1.0'])

# Build summary dataframe
fraud_by_bucket = fraud_score.groupby('score_bucket', observed=True).agg(
    fraud_count  = ('isFraud',      'sum'),
    fraud_volume = ('fraud_amount', 'sum')
).reset_index()

# % of total fraud count and volume
fraud_by_bucket['pct_fraud_count']  = (fraud_by_bucket['fraud_count']  / fraud_by_bucket['fraud_count'].sum()  * 100).round(2)
fraud_by_bucket['pct_fraud_volume'] = (fraud_by_bucket['fraud_volume'] / fraud_by_bucket['fraud_volume'].sum() * 100).round(2)

fraud_by_bucket.sort_values(by='score_bucket', inplace=True)
fraud_by_bucket

,score_bucket,fraud_count,fraud_volume,pct_fraud_count,pct_fraud_volume
0,0-0.1,43,5.201727e+06,1.72,0.14
1,0.1-0.2,43,3.266912e+06,1.72,0.09
2,0.2-0.3,21,1.622831e+06,0.84,0.04
3,0.3-0.4,29,3.768571e+06,1.16,0.10
4,0.4-0.5,27,5.638568e+06,1.08,0.16
5,0.5-0.6,39,7.411837e+06,1.56,0.20
6,0.6-0.7,37,5.279440e+06,1.48,0.15
7,0.7-0.8,57,9.626188e+06,2.28,0.26
8,0.8-0.9,99,1.632436e+07,3.97,0.45
9,0.9-1.0,2101,3.577976e+09,84.17,98.40
